# AWS Mushroom — correct cursusbestand (5.000 rijen)

Jorre Van Dyck: AWS-training; Codex: voorbereid vervangingsnotebook op verzoek van
Jorre, 14 oktober 2026. **Nog niet op SageMaker uitgevoerd.** Upload het CSV-bestand
uit `MushroomFinal`. Dit notebook vervangt voor de inlevering de eerdere training op
61.069 UCI-rijen. `jumbled_noise_0` wordt uitgesloten; `jumbled_noise_1` blijft behouden.
Het notebook maakt geen endpoint aan.

## 1. Upload en controleer de juiste bron
Upload mushroom_project_dataset.csv naar dezelfde Jupyter-map als dit notebook. De bronhash voorkomt dat de verkeerde dataset wordt gebruikt.

In [ ]:
from pathlib import Path
import hashlib, json, platform
import numpy as np
import pandas as pd
import sklearn, joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
source=Path('mushroom_project_dataset.csv')
if not source.exists():
    source=Path('../../MushroomFinal/mushroom_project_dataset.csv')
source_hash=hashlib.sha256(source.read_bytes().replace(b'\r\n',b'\n')).hexdigest()
assert source_hash=='1f1a25f2f330458ed95ce9f6ffe3241582312a42cb21797a2b7bd5e5ad281114'
data=pd.read_csv(source)
assert data.shape==(5000,13)
data.columns=data.columns.str.strip().str.lower().str.replace('-','_',regex=False)
X=data.drop(columns=['class','jumbled_noise_0'])
y=data['class'].map({'e':0,'p':1})
assert X.shape==(5000,11) and 'jumbled_noise_1' in X
print(X.shape, y.value_counts())

## 2. Dezelfde splits en volledige pipeline
Gebruik dezelfde 4.100/900 rijen en vijf trainingsfolds als de teamvergelijking. Imputatie en encoding worden per trainingsfold geleerd.

In [ ]:
train,test=train_test_split(np.arange(len(X)),test_size=.18,random_state=42,stratify=y)
assert len(train)==4100 and len(test)==900
numeric=X.select_dtypes(include='number').columns.tolist()
categorical=X.select_dtypes(exclude='number').columns.tolist()
preprocessor=ColumnTransformer([
    ('num',SimpleImputer(strategy='median'),numeric),
    ('cat',Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='missing')),
        ('encode',OneHotEncoder(handle_unknown='ignore'))]),categorical)])
pipeline=Pipeline([('preprocessor',preprocessor),('classifier',RandomForestClassifier(random_state=42,n_jobs=2))])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
# Four declared candidates; select only on training CV macro F1.
parameters=[
    {'classifier__n_estimators':[100],'classifier__max_depth':[None],'classifier__min_samples_leaf':[1]},
    {'classifier__n_estimators':[100],'classifier__max_depth':[24],'classifier__min_samples_leaf':[2]},
    {'classifier__n_estimators':[200],'classifier__max_depth':[16],'classifier__min_samples_leaf':[2]},
    {'classifier__n_estimators':[200],'classifier__max_depth':[None],'classifier__min_samples_leaf':[1]}]
search=GridSearchCV(pipeline,parameters,scoring='f1_macro',cv=cv,n_jobs=2,refit=True)
search.fit(X.iloc[train],y.iloc[train])
print(search.best_params_,search.best_score_)

## 3. Evalueer en exporteer
De testset is al bekend uit het teamonderzoek; benoem dit als ontwikkeltest. Download na uitvoering deze notebook en de drie correct_*-exports.

In [ ]:
model=search.best_estimator_
pred=model.predict(X.iloc[test]);prob=model.predict_proba(X.iloc[test])[:,list(model.classes_).index(1)]
results={'source':'mushroom_project_dataset.csv','source_sha256_lf':source_hash,
    'rows':5000,'train_rows':4100,'test_rows':900,'features':X.columns.tolist(),
    'removed':['jumbled_noise_0'],'class_encoding':{'e':0,'p':1},
    'best_parameters':search.best_params_,'cv_macro_f1':float(search.best_score_),
    'accuracy':float(accuracy_score(y.iloc[test],pred)),
    'balanced_accuracy':float(balanced_accuracy_score(y.iloc[test],pred)),
    'macro_f1':float(f1_score(y.iloc[test],pred,average='macro')),
    'precision_p':float(precision_score(y.iloc[test],pred,zero_division=0)),
    'recall_p':float(recall_score(y.iloc[test],pred,zero_division=0)),
    'f1_p':float(f1_score(y.iloc[test],pred,zero_division=0)),
    'roc_auc':float(roc_auc_score(y.iloc[test],prob)),
    'confusion_matrix':confusion_matrix(y.iloc[test],pred,labels=[0,1]).tolist(),
    'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__,'pandas':pd.__version__,'numpy':np.__version__,'joblib':joblib.__version__},
    'execution_environment':'AWS SageMaker - record only after actual execution'}
Path('correct_aws_results.json').write_text(json.dumps(results,indent=2)+'\n')
pd.DataFrame(search.cv_results_).to_csv('correct_aws_tuning_results.csv',index=False)
joblib.dump(model,'correct_mushroom_pipeline.joblib',compress=3)
print(json.dumps(results,indent=2))